In [ ]:
#coding=utf-8
import os

import numpy as np
import torch
from PIL import Image
from matplotlib import pyplot as plt

from common import (
    get_fused_output,
    load_hierarchy_info,
    process_image,
    resnet50,
    seed_everything,
    set_gpu_envs,
)


In [ ]:
#######################parameter settings##############################
setname = 'CUB_H' #'CUB_H', 'Butterfly_H'
CUDA_VISIBLE_DEVICES = [0]
input_size = 224 #224,448
transfer_mode_test = 'center'

model_name = 'resnet-50-coarse-fine-size-'
model_name += str(input_size) + '-hier'
seed_everything()
gpu_envs_str = set_gpu_envs(CUDA_VISIBLE_DEVICES)
softmax = torch.nn.Softmax(dim=1)

#######################load the hierarchy info########################
hierarchy_info = load_hierarchy_info(setname)
label_len = hierarchy_info['label_len']
trans_1_to_2 = torch.from_numpy(hierarchy_info['trans_1_to_2']).type(torch.float32)
trans_2_to_3 = torch.from_numpy(hierarchy_info['trans_2_to_3']).type(torch.float32)
trans_3_to_4 = torch.from_numpy(hierarchy_info['trans_3_to_4']).type(torch.float32)
trans_1_to_4 = torch.matmul(trans_1_to_2, torch.matmul(trans_2_to_3, trans_3_to_4))
trans_2_to_4 = torch.matmul(trans_2_to_3, trans_3_to_4)
print(label_len)

#######################load the model##################################
model_path = './experiments/'+setname+'/'+model_name+'/code-20230816-183810/last-model.pth'
model = resnet50(num_classes=label_len)
model.load_state_dict(torch.load(model_path)['model_state_dict'])
print('Load the trained model parameters success!')
#model.cuda()
model.eval()

#####################read the test images#######################
test_dir = './test'
listdir = os.listdir(test_dir)

for idx in range(len(listdir)):
    onefile = test_dir+'/'+ listdir[idx]
    print(onefile)
    oneimg_pil = Image.open(onefile)
    imgdata = process_image(oneimg_pil, input_size, transfer_mode_test)
    imgdata = imgdata.float().unsqueeze(0)
    outputs_list = model(imgdata)
    output_v1,output_v2,output_v3,output_v4 = outputs_list
    output_merged = get_fused_output(outputs_list, trans_1_to_4,trans_2_to_4,trans_3_to_4)
    predict_v1 = output_v1.max(1)[1][0].numpy().astype(np.int32).tolist()
    predict_v2 = output_v2.max(1)[1][0].numpy().astype(np.int32).tolist()
    predict_v3 = output_v3.max(1)[1][0].numpy().astype(np.int32).tolist()
    predict_v4 = output_v4.max(1)[1][0].numpy().astype(np.int32).tolist()
    print('predicted label is: ', [predict_v1,predict_v2,predict_v3,predict_v4])
    plt.imshow(np.asarray(oneimg_pil))
    plt.show()
